# LLM Results Analysis

This notebook processes raw experiment results and computes summary statistics.

## Input
- Raw iteration results: `{task}_{model}_ft_{finetune}_iter_{i}.csv`
- Embedding regression results

## Output
- Summary CSVs with mode predictions and entropy
- Comparison tables across models and tasks

## 1. Setup

In [ ]:
import os
import pandas as pd
import numpy as np
from collections import Counter
from scipy.stats import entropy
from sklearn.metrics import mean_squared_error, accuracy_score, mean_absolute_error
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Configuration
RESULTS_DIR = '../results'
SUMMARY_DIR = '../results/summaries'
os.makedirs(SUMMARY_DIR, exist_ok=True)

TASKS = ['Bandgap', 'Dielectric', 'CrystalStructure', 'MatKG']

MODELS = [
    'gpt-4o-mini', 'gpt-4o', 'gpt-4.1', 'gpt-4.1-mini', 'gpt-3.5',
    'deepseek-r1-distill-qwen-32b',
    'llama-2-7b-chat', 'llama-2-13b-chat', 'llama-2-70b-chat',
    'llama-3-8b-instruct', 'llama-3-70b-instruct',
    'mistral-7b-instruct', 'mistral-7b-instruct-v0-2', 'mistral-7b-instruct-v0-3',
    'mixtral-8x7b-instruct-v0-1',
    'zephyr-7b-beta',
    'codellama-7b-instruct', 'codellama-13b-instruct', 'codellama-70b-instruct',
    'gemma-2b-instruct', 'gemma-7b-instruct',
    'phi-3-mini-4k-instruct'
]

FINETUNE_CONDITIONS = ['base', 'Bandgap', 'Dielectric', 'CrystalStructure', 'MatKG']

## 2. Processing Functions

In [ ]:
def aggregate_iterations(task, model, finetune_on, n_iterations=10):
    """
    Aggregate results from multiple iterations into a summary.
    
    Computes:
    - Individual predictions per iteration
    - Mode prediction (median for regression, mode for classification)
    - Prediction entropy (consistency measure)
    """
    all_preds = {}
    
    for i in range(n_iterations):
        filepath = os.path.join(RESULTS_DIR, f'{task}_{model}_ft_{finetune_on}_iter_{i}.csv')
        if not os.path.exists(filepath):
            continue
        
        df = pd.read_csv(filepath, index_col=0)
        
        for idx, row in df.iterrows():
            if idx not in all_preds:
                all_preds[idx] = {
                    'Formula': row.get('Formula', ''),
                    'Ground': row.get('Ground', None),
                    'predictions': []
                }
            all_preds[idx]['predictions'].append(row.get('Prediction', None))
    
    if not all_preds:
        return None
    
    # Create summary
    records = []
    for mpid, data in all_preds.items():
        record = {
            'mpid': mpid,
            'Formula': data['Formula'],
            'Ground': data['Ground']
        }
        
        # Add individual predictions
        preds = data['predictions']
        for i, p in enumerate(preds):
            record[f'prediction_{i+1}'] = p
        
        # Filter valid predictions
        valid_preds = [p for p in preds if p is not None and not pd.isna(p)]
        
        if valid_preds:
            if task in ['Bandgap', 'Dielectric']:
                # Regression: use median
                valid_nums = [float(p) for p in valid_preds if isinstance(p, (int, float)) or str(p).replace('.', '').replace('-', '').isdigit()]
                if valid_nums:
                    record['mode_prediction'] = np.median(valid_nums)
                    # Entropy via binning
                    if len(valid_nums) > 1:
                        bins = np.histogram_bin_edges(valid_nums, bins='auto')
                        hist, _ = np.histogram(valid_nums, bins=bins)
                        record['prediction_entropy'] = entropy(hist + 1e-10)
                    else:
                        record['prediction_entropy'] = 0
            else:
                # Classification: use mode
                counter = Counter(valid_preds)
                record['mode_prediction'] = counter.most_common(1)[0][0]
                counts = list(counter.values())
                record['prediction_entropy'] = entropy(counts) if len(counts) > 1 else 0
        
        records.append(record)
    
    return pd.DataFrame(records)

In [ ]:
def compute_metrics(df, task):
    """
    Compute evaluation metrics from a summary DataFrame.
    """
    if 'mode_prediction' not in df.columns or 'Ground' not in df.columns:
        return None
    
    # Drop rows with missing predictions
    valid_df = df.dropna(subset=['mode_prediction', 'Ground'])
    
    if len(valid_df) == 0:
        return None
    
    if task in ['Bandgap', 'Dielectric']:
        # Regression metrics
        y_true = valid_df['Ground'].astype(float)
        y_pred = valid_df['mode_prediction'].astype(float)
        
        return {
            'n_samples': len(valid_df),
            'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
            'mae': mean_absolute_error(y_true, y_pred),
            'mean_entropy': df['prediction_entropy'].mean() if 'prediction_entropy' in df.columns else None
        }
    else:
        # Classification metrics
        y_true = valid_df['Ground'].astype(str)
        y_pred = valid_df['mode_prediction'].astype(str)
        
        return {
            'n_samples': len(valid_df),
            'accuracy': accuracy_score(y_true, y_pred) * 100,
            'mean_entropy': df['prediction_entropy'].mean() if 'prediction_entropy' in df.columns else None
        }

## 3. Process All Results

In [ ]:
def process_all_results():
    """
    Process all raw results into summaries.
    """
    all_metrics = []
    
    for task in tqdm(TASKS, desc="Tasks"):
        for model in MODELS:
            for ft_condition in FINETUNE_CONDITIONS:
                output_file = os.path.join(SUMMARY_DIR, f'{task}_{model}_ft_{ft_condition}_summary.csv')
                
                # Check if summary already exists
                if os.path.exists(output_file):
                    summary_df = pd.read_csv(output_file)
                else:
                    # Aggregate from iterations
                    summary_df = aggregate_iterations(task, model, ft_condition)
                    
                    if summary_df is not None:
                        summary_df.to_csv(output_file, index=False)
                
                if summary_df is not None:
                    metrics = compute_metrics(summary_df, task)
                    if metrics:
                        metrics.update({
                            'task': task,
                            'model': model,
                            'finetune_on': ft_condition
                        })
                        all_metrics.append(metrics)
    
    return pd.DataFrame(all_metrics)

# Process all
# metrics_df = process_all_results()

## 4. Load Existing Summaries

In [ ]:
def load_all_summaries(result_dir):
    """
    Load all existing summary files and compute metrics.
    """
    all_metrics = []
    
    for filename in os.listdir(result_dir):
        if not filename.endswith('_summary.csv'):
            continue
        
        # Parse filename
        parts = filename.replace('_summary.csv', '').split('_ft_')
        if len(parts) != 2:
            continue
        
        task_model = parts[0]
        ft_condition = parts[1]
        
        # Extract task and model
        for task in TASKS:
            if task_model.startswith(task + '_'):
                model = task_model[len(task)+1:]
                break
        else:
            continue
        
        # Load and compute metrics
        filepath = os.path.join(result_dir, filename)
        df = pd.read_csv(filepath)
        
        metrics = compute_metrics(df, task)
        if metrics:
            metrics.update({
                'task': task,
                'model': model,
                'finetune_on': ft_condition,
                'filepath': filepath
            })
            all_metrics.append(metrics)
    
    return pd.DataFrame(all_metrics)

# Load all summaries
# metrics_df = load_all_summaries(RESULTS_DIR)

## 5. Analysis Tables

In [ ]:
def create_comparison_table(metrics_df, task, metric='rmse'):
    """
    Create a comparison table: models (rows) vs finetune conditions (columns).
    """
    task_df = metrics_df[metrics_df['task'] == task]
    
    pivot = task_df.pivot(
        index='model',
        columns='finetune_on',
        values=metric
    )
    
    # Reorder columns
    col_order = ['base'] + [t for t in TASKS if t in pivot.columns]
    pivot = pivot.reindex(columns=[c for c in col_order if c in pivot.columns])
    
    return pivot

def compute_improvement(metrics_df, task):
    """
    Compute improvement from base to finetuned (same task).
    """
    task_df = metrics_df[metrics_df['task'] == task]
    
    improvements = []
    
    for model in task_df['model'].unique():
        model_df = task_df[task_df['model'] == model]
        
        base_row = model_df[model_df['finetune_on'] == 'base']
        ft_row = model_df[model_df['finetune_on'] == task]
        
        if len(base_row) > 0 and len(ft_row) > 0:
            if task in ['Bandgap', 'Dielectric']:
                base_val = base_row['rmse'].values[0]
                ft_val = ft_row['rmse'].values[0]
                improvement = (base_val - ft_val) / base_val * 100
            else:
                base_val = base_row['accuracy'].values[0]
                ft_val = ft_row['accuracy'].values[0]
                improvement = ft_val - base_val
            
            improvements.append({
                'model': model,
                'base': base_val,
                'finetuned': ft_val,
                'improvement': improvement
            })
    
    return pd.DataFrame(improvements).sort_values('finetuned', ascending=(task in ['Bandgap', 'Dielectric']))

## 6. Cross-Task Transfer Analysis

In [ ]:
def analyze_cross_task_transfer(metrics_df, eval_task):
    """
    Analyze how finetuning on different tasks affects performance on eval_task.
    """
    task_df = metrics_df[metrics_df['task'] == eval_task]
    
    # Group by finetune condition
    metric = 'rmse' if eval_task in ['Bandgap', 'Dielectric'] else 'accuracy'
    
    summary = task_df.groupby('finetune_on')[metric].agg(['mean', 'std', 'count']).reset_index()
    summary.columns = ['finetune_on', 'mean', 'std', 'n_models']
    
    # Sort by performance
    ascending = eval_task in ['Bandgap', 'Dielectric']  # Lower RMSE is better
    summary = summary.sort_values('mean', ascending=ascending)
    
    return summary

# Example usage:
# for task in TASKS:
#     print(f"\n{task}:")
#     print(analyze_cross_task_transfer(metrics_df, task))

## 7. Model Rankings

In [ ]:
def rank_models(metrics_df, task, finetune_on='same'):
    """
    Rank models by performance.
    
    Args:
        finetune_on: 'same' (same task), 'base', or specific task name
    """
    task_df = metrics_df[metrics_df['task'] == task]
    
    if finetune_on == 'same':
        task_df = task_df[task_df['finetune_on'] == task]
    else:
        task_df = task_df[task_df['finetune_on'] == finetune_on]
    
    metric = 'rmse' if task in ['Bandgap', 'Dielectric'] else 'accuracy'
    ascending = task in ['Bandgap', 'Dielectric']
    
    ranked = task_df.sort_values(metric, ascending=ascending)[['model', metric, 'mean_entropy']]
    ranked['rank'] = range(1, len(ranked) + 1)
    
    return ranked

# Example:
# print(rank_models(metrics_df, 'Bandgap', 'same').head(10))

## 8. Export Results

In [ ]:
def export_analysis_results(metrics_df, output_dir):
    """
    Export all analysis results to CSV files.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Full metrics
    metrics_df.to_csv(os.path.join(output_dir, 'all_metrics.csv'), index=False)
    
    # Per-task comparison tables
    for task in TASKS:
        metric = 'rmse' if task in ['Bandgap', 'Dielectric'] else 'accuracy'
        table = create_comparison_table(metrics_df, task, metric)
        table.to_csv(os.path.join(output_dir, f'{task}_comparison.csv'))
        
        # Improvement table
        improvement = compute_improvement(metrics_df, task)
        improvement.to_csv(os.path.join(output_dir, f'{task}_improvement.csv'), index=False)
    
    # Cross-task transfer
    transfer_results = []
    for task in TASKS:
        summary = analyze_cross_task_transfer(metrics_df, task)
        summary['eval_task'] = task
        transfer_results.append(summary)
    
    pd.concat(transfer_results).to_csv(os.path.join(output_dir, 'cross_task_transfer.csv'), index=False)
    
    print(f"Results exported to {output_dir}")

# Export
# export_analysis_results(metrics_df, '../results/analysis')

## Notes

### File Naming Convention
- Raw: `{task}_{model}_ft_{finetune}_iter_{i}.csv`
- Summary: `{task}_{model}_ft_{finetune}_summary.csv`

### Metrics
- **Bandgap/Dielectric**: RMSE, MAE (lower is better)
- **CrystalStructure**: Accuracy % (higher is better)
- **MatKG**: Top-1/Top-5 Accuracy % (higher is better)

### Entropy
- Lower entropy = more consistent predictions across iterations
- Higher entropy = more variability in predictions